In [2]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, precision_recall_curve, auc,
    roc_auc_score, confusion_matrix
)

DATA_PATH = "financial_transactions_synthetic.csv"
MODEL_DIR = "model_artifacts"
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. Load
df = pd.read_csv(DATA_PATH)
print("Loaded:", df.shape)

# 2. Feature engineering
df["transaction_datetime"] = pd.to_datetime(df["transaction_date"] + " " + df["transaction_time"])
df["transaction_hour"] = df["transaction_datetime"].dt.hour
df["transaction_dayofweek"] = df["transaction_datetime"].dt.dayofweek
df["is_night"] = df["transaction_hour"].apply(lambda h: 1 if (h < 5 or h == 23) else 0)

df["amount_vs_avg"] = df["transaction_amount"] / df["average_transaction_amount"].replace(0, 1)
df["amount_vs_prev"] = df["transaction_amount"] / df["previous_transaction_amount"].replace(0, 1)
df["location_mismatch"] = (df["customer_location"] != df["merchant_location"]).astype(int)

# 3. Encode categoricals
categorical_cols = ["transaction_type", "payment_method", "currency", "account_type",
                     "merchant_category", "device_type"]
encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

FEATURE_COLS = [
    "transaction_amount", "customer_age", "account_age_days",
    "previous_transaction_amount", "average_transaction_amount",
    "transaction_frequency", "international_transaction",
    "failed_transaction_count", "transaction_velocity",
    "unusual_location", "new_device",
    "transaction_hour", "transaction_dayofweek", "is_night",
    "amount_vs_avg", "amount_vs_prev", "location_mismatch",
] + [c + "_enc" for c in categorical_cols]

TARGET_COL = "fraud_label"
X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

# 4. Train/test split (stratified — fraud is rare)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 5. Scale (used by IsolationForest)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Classifier
clf = RandomForestClassifier(
    n_estimators=300, max_depth=14, min_samples_leaf=5,
    class_weight="balanced_subsample", n_jobs=-1, random_state=42,
)
clf.fit(X_train, y_train)

y_proba = clf.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, digits=4))
precision, recall, _ = precision_recall_curve(y_test, y_proba)
print(f"PR-AUC:  {auc(recall, precision):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

# 7. Anomaly detector — trained only on legitimate transactions
iso = IsolationForest(n_estimators=200, contamination=0.01, random_state=42, n_jobs=-1)
iso.fit(X_train_scaled[y_train.values == 0])

# 8. Save artifacts
joblib.dump(clf, f"{MODEL_DIR}/fraud_classifier.pkl")
joblib.dump(iso, f"{MODEL_DIR}/anomaly_detector.pkl")
joblib.dump(scaler, f"{MODEL_DIR}/scaler.pkl")
joblib.dump(encoders, f"{MODEL_DIR}/encoders.pkl")
joblib.dump(FEATURE_COLS, f"{MODEL_DIR}/feature_cols.pkl")
print("\nSaved model artifacts to:", MODEL_DIR)

Loaded: (400000, 25)

=== Classification Report ===
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000     79040
           1     1.0000    1.0000    1.0000       960

    accuracy                         1.0000     80000
   macro avg     1.0000    1.0000    1.0000     80000
weighted avg     1.0000    1.0000    1.0000     80000

PR-AUC:  1.0000
ROC-AUC: 1.0000
Confusion matrix:
 [[79040     0]
 [    0   960]]

Saved model artifacts to: model_artifacts


In [3]:
%%writefile fraud_detection_app.py
import streamlit as st
import pandas as pd
from datetime import datetime
import joblib

MODEL_DIR = "model_artifacts"

@st.cache_resource
def load_artifacts():
    clf = joblib.load(f"{MODEL_DIR}/fraud_classifier.pkl")
    iso = joblib.load(f"{MODEL_DIR}/anomaly_detector.pkl")
    scaler = joblib.load(f"{MODEL_DIR}/scaler.pkl")
    encoders = joblib.load(f"{MODEL_DIR}/encoders.pkl")
    feature_cols = joblib.load(f"{MODEL_DIR}/feature_cols.pkl")
    return clf, iso, scaler, encoders, feature_cols

clf, iso, scaler, encoders, FEATURE_COLS = load_artifacts()

def safe_encode(encoder, value):
    if value in encoder.classes_:
        return encoder.transform([value])[0]
    return encoder.transform([encoder.classes_[0]])[0]

def risk_level_from_score(score):
    if score < 40:
        return "LOW", "🟢"
    elif score < 70:
        return "MEDIUM", "🟡"
    return "HIGH", "🔴"

st.set_page_config(page_title="Fraud Detection Simulator", page_icon="🕵️", layout="centered")
st.title("🕵️ Real-Time Transaction Fraud Simulator")

with st.form("transaction_form"):
    col1, col2 = st.columns(2)
    with col1:
        transaction_amount = st.number_input("Transaction Amount (₹)", min_value=1.0, value=85000.0, step=100.0)
        transaction_type = st.selectbox("Transaction Type", ["online", "in_person"])
        payment_method = st.selectbox("Payment Method", ["credit_card", "debit_card", "upi", "net_banking", "wallet"])
        currency = st.selectbox("Currency", ["INR", "USD"])
        merchant_category = st.selectbox("Merchant Category",
            ["grocery","electronics","online_retail","travel","restaurant","fuel",
             "entertainment","healthcare","fashion","utilities","gaming","jewelry",
             "crypto_exchange","money_transfer"])
        device_type = st.selectbox("Device Type", ["mobile", "desktop", "tablet", "pos_terminal"])
        international_transaction = st.selectbox("International Transaction?", ["No", "Yes"])
    with col2:
        customer_age = st.number_input("Customer Age", min_value=18, max_value=100, value=34)
        account_type = st.selectbox("Account Type", ["savings", "current", "premium"])
        account_age_days = st.number_input("Account Age (days)", min_value=1, value=800)
        average_transaction_amount = st.number_input("Customer's Avg Transaction (₹)", min_value=1.0, value=3500.0)
        previous_transaction_amount = st.number_input("Previous Transaction Amount (₹)", min_value=0.0, value=3200.0)
        transaction_frequency = st.number_input("Transactions in last 24h", min_value=0, value=2)
        failed_transaction_count = st.number_input("Failed Attempts (recent)", min_value=0, value=0)
        new_device = st.selectbox("New / Unrecognized Device?", ["No", "Yes"])
        unusual_location = st.selectbox("Unusual Location for this Customer?", ["No", "Yes"])
    transaction_time = st.time_input("Transaction Time", value=datetime.now().time())
    submitted = st.form_submit_button("🔍 Analyze Transaction", use_container_width=True)

if submitted:
    hour = transaction_time.hour
    is_night = 1 if (hour < 5 or hour == 23) else 0
    amount_vs_avg = transaction_amount / max(average_transaction_amount, 1)
    amount_vs_prev = transaction_amount / max(previous_transaction_amount, 1)
    location_mismatch = 1 if unusual_location == "Yes" else 0

    row = {
        "transaction_amount": transaction_amount,
        "customer_age": customer_age,
        "account_age_days": account_age_days,
        "previous_transaction_amount": previous_transaction_amount,
        "average_transaction_amount": average_transaction_amount,
        "transaction_frequency": transaction_frequency,
        "international_transaction": 1 if international_transaction == "Yes" else 0,
        "failed_transaction_count": failed_transaction_count,
        "transaction_velocity": transaction_frequency,
        "unusual_location": 1 if unusual_location == "Yes" else 0,
        "new_device": 1 if new_device == "Yes" else 0,
        "transaction_hour": hour,
        "transaction_dayofweek": datetime.now().weekday(),
        "is_night": is_night,
        "amount_vs_avg": amount_vs_avg,
        "amount_vs_prev": amount_vs_prev,
        "location_mismatch": location_mismatch,
        "transaction_type_enc": safe_encode(encoders["transaction_type"], transaction_type),
        "payment_method_enc": safe_encode(encoders["payment_method"], payment_method),
        "currency_enc": safe_encode(encoders["currency"], currency),
        "account_type_enc": safe_encode(encoders["account_type"], account_type),
        "merchant_category_enc": safe_encode(encoders["merchant_category"], merchant_category),
        "device_type_enc": safe_encode(encoders["device_type"], device_type),
    }

    X_input = pd.DataFrame([row])[FEATURE_COLS]
    fraud_proba = clf.predict_proba(X_input)[0, 1]
    risk_score = round(fraud_proba * 100, 1)
    risk_level, emoji = risk_level_from_score(risk_score)

    X_scaled = scaler.transform(X_input)
    is_anomaly = iso.predict(X_scaled)[0] == -1
    prediction_label = "FRAUDULENT" if fraud_proba >= 0.5 else "LEGITIMATE"

    st.divider()
    if prediction_label == "FRAUDULENT":
        st.error(f"🚨 Prediction: **{prediction_label}**")
    else:
        st.success(f"✅ Prediction: **{prediction_label}**")

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Fraud Probability", f"{fraud_proba*100:.1f}%")
    c2.metric("Risk Score", f"{risk_score}/100")
    c3.metric("Risk Level", f"{emoji} {risk_level}")
    c4.metric("Anomaly", "YES" if is_anomaly else "NO")

Writing fraud_detection_app.py
